# 02 — The Agent Card: How Agents Describe Themselves

## Why this notebook exists

In **notebook 01** we built two services that talked to each other through hand-rolled REST. We watched that fall over the moment one side changed its contract — because there was no machine-readable description of either service. Consumers had no way to ask: *"what skills do you have, what inputs do you take, and how do I authenticate?"*

A2A solves this with a single, simple primitive: the **Agent Card**. It's a JSON document served at a well-known URL (`/.well-known/agent.json`) that describes everything a client needs to know to start talking to an agent.

This notebook builds an Agent Card for the researcher from notebook 01, fetches it from a client, and walks through what the card actually says.

## What you'll learn

- The shape of an A2A **Agent Card**: name, description, URL, capabilities, authentication, skills.
- How to serve `/.well-known/agent.json` from a FastAPI app.
- How to discover an agent's capabilities from the client side using `httpx`.
- How to parse a card into a typed `pydantic` model and iterate over its declared skills.
- Why the Agent Card is **not** the same as OpenAPI, and what each is good for.

## 1. Setup

Same pattern as notebook 01 — we run a FastAPI app on a background thread so we can both serve and call it from this notebook. Each notebook in this series is self-contained, so the helper is re-defined here rather than imported.

In [ ]:
import json
import threading
import time

import httpx
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    """Start `app` on localhost:`port` in a background daemon thread."""
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")

    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


print("Setup OK")